In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [102]:
df = pd.read_csv("/content/cyber_attack_dataset_100000.csv")
df.head()

,duration,src_bytes,dst_bytes,packet_count,protocol,failed_logins,attack_type
0,1,8605,418,631,TCP,0,DDoS
1,1,499,148,131,UDP,0,PortScan
2,10,370,160,105,UDP,0,PortScan
3,2,5138,320,666,TCP,0,DDoS
4,36,524,467,58,UDP,10,BruteForce


In [103]:
df.describe()

,duration,src_bytes,dst_bytes,packet_count,failed_logins
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000
mean,16.462720,2219.62870,426.171030,202.550770,1.630520
std,15.632207,2751.14241,466.528205,249.729802,3.045577
min,1.000000,50.00000,20.000000,5.000000,0.000000
25%,4.000000,341.00000,131.000000,37.000000,0.000000
50%,10.000000,781.00000,256.000000,87.000000,0.000000
75%,28.000000,3024.00000,458.000000,257.000000,3.000000
max,60.000000,10000.00000,2000.000000,1000.000000,10.000000


In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   duration       100000 non-null  int64 
 1   src_bytes      100000 non-null  int64 
 2   dst_bytes      100000 non-null  int64 
 3   packet_count   100000 non-null  int64 
 4   protocol       100000 non-null  object
 5   failed_logins  100000 non-null  int64 
 6   attack_type    100000 non-null  object
dtypes: int64(5), object(2)
memory usage: 5.3+ MB


In [105]:
df.isnull().sum()

,0
duration,0
src_bytes,0
dst_bytes,0
packet_count,0
protocol,0
failed_logins,0
attack_type,0


In [106]:
df.shape

(100000, 7)

In [107]:
df['Data_transfer_ratio'] = df['src_bytes'] /df['dst_bytes']

#df['traffic_ratio'] = (df['src_bytes'] /(df['dst_bytes'] + 1))

#df['packets_per_second'] = (df['packet_count'] /(df['duration'] + 1))

df['bytes_per_second'] = ((df['src_bytes'] + df['dst_bytes']) /(df['duration'] + 1))

df['login_success_rate'] = (df['failed_logins'] /(df['packet_count'] ))

In [108]:
df = df.drop("duration" , axis=1)
df =df.drop("src_bytes" , axis=1)
df =df.drop("dst_bytes" , axis=1)
df =df.drop("packet_count" , axis=1)
df =df.drop("failed_logins" , axis=1)

In [109]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["attack_type"]=le.fit_transform(df["attack_type"])

In [110]:
df = pd.get_dummies(df,columns=["protocol"])

In [111]:
df.head()

,attack_type,Data_transfer_ratio,bytes_per_second,login_success_rate,protocol_TCP,protocol_UDP
0,1,20.586124,4511.500000,0.000000,True,False
1,3,3.371622,323.500000,0.000000,False,True
2,3,2.312500,48.181818,0.000000,False,True
3,1,16.056250,1819.333333,0.000000,True,False
4,0,1.122056,26.783784,0.172414,False,True


In [112]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(df[['protocol_TCP']])

encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(['protocol_TCP']))

df = df.drop('protocol_TCP', axis=1)
df = pd.concat([df, encoded_df], axis=1)

In [113]:
df.head()

,attack_type,Data_transfer_ratio,bytes_per_second,login_success_rate,protocol_UDP,protocol_TCP_False,protocol_TCP_True
0,1,20.586124,4511.500000,0.000000,False,0.0,1.0
1,3,3.371622,323.500000,0.000000,True,1.0,0.0
2,3,2.312500,48.181818,0.000000,True,1.0,0.0
3,1,16.056250,1819.333333,0.000000,False,0.0,1.0
4,0,1.122056,26.783784,0.172414,True,1.0,0.0


In [114]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(df[['protocol_UDP']])

encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(['protocol_UDP']))

df = df.drop('protocol_UDP', axis=1)
df = pd.concat([df, encoded_df], axis=1)

In [115]:
df.head()

,attack_type,Data_transfer_ratio,bytes_per_second,login_success_rate,protocol_TCP_False,protocol_TCP_True,protocol_UDP_False,protocol_UDP_True
0,1,20.586124,4511.500000,0.000000,0.0,1.0,1.0,0.0
1,3,3.371622,323.500000,0.000000,1.0,0.0,0.0,1.0
2,3,2.312500,48.181818,0.000000,1.0,0.0,0.0,1.0
3,1,16.056250,1819.333333,0.000000,0.0,1.0,1.0,0.0
4,0,1.122056,26.783784,0.172414,1.0,0.0,0.0,1.0


In [116]:
X = df.drop("attack_type" , axis=1)
y = df["attack_type"]

In [117]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [118]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [119]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , recall_score , precision_score , f1_score , confusion_matrix

In [120]:
model = XGBClassifier(
    random_state=42,
)

In [121]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [122]:
y_pred = model.predict(X_test)

In [123]:
print("accuracy :" , accuracy_score(y_test , y_pred))
print("recall :" , recall_score(y_test , y_pred , average='weighted'))
print("precision :" , precision_score(y_test , y_pred , average='weighted'))
print("f1 :" , f1_score(y_test , y_pred , average='weighted'))

accuracy : 0.84165
recall : 0.84165
precision : 0.846635205355435
f1 : 0.8397989920122021
